# Silver Data: Products + EPD
File này xử lý cả 2 nguồn dữ liệu sản phẩm:
- `products.csv` (nguồn chính)
- `epd.json` (nguồn bổ sung)

**Logic gộp:**
1. Đọc & làm sạch cả 2 nguồn
2. Chuẩn hoá kiểu dữ liệu trước khi so sánh
3. So sánh TẤT CẢ các cột (trừ `product_id`)
4. Sản phẩm EPD đã có trong Products → bỏ qua
5. Chỉ thêm sản phẩm thật sự mới từ EPD
6. Xuất ra 1 file CSV duy nhất

In [1]:
import pandas as pd
import numpy as np


## PHẦN 1: Đọc & Làm sạch Products

In [2]:
products = pd.read_csv('../products.csv')
display(products.head())
products.info()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   object 
 2   category      2412 non-null   object 
 3   segment       2412 non-null   object 
 4   size          2412 non-null   object 
 5   color         2412 non-null   object 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 150.9+ KB


In [3]:
# Theo Dictionary, product_id là PK => kiểm tra trùng lặp
print('product_id là unique:', products['product_id'].is_unique)

product_id là unique: True


In [4]:
# Xem các giá trị category
products['category'].value_counts()

category
Streetwear    1320
Outdoor        743
Casual         201
GenZ           148
Name: count, dtype: int64

In [5]:
# Xem các giá trị segment
products['segment'].value_counts()

segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148
Name: count, dtype: int64

In [6]:
# Xem các giá trị size
products['size'].value_counts()

size
S     603
M     603
L     603
XL    603
Name: count, dtype: int64

In [7]:
# Xem các giá trị color
products['color'].value_counts()

color
black     242
orange    242
green     241
silver    241
pink      241
yellow    241
red       241
blue      241
white     241
purple    241
Name: count, dtype: int64

In [8]:
# Products đã sạch: không null, không trùng, các giá trị hợp lệ
print(f'✅ Products sạch: {len(products)} dòng, {products.isna().sum().sum()} null')

✅ Products sạch: 2412 dòng, 0 null


## PHẦN 2: Đọc & Làm sạch EPD

In [9]:
epd = pd.read_json('../epd.json')
display(epd.head())
epd.info()

,product_id,product_name,category,segment,size,color,price,cogs
0,1,BambooCraft UC-21,Streetwear,Everyday,L,orange,19529.37,16287.494580
1,2,MekongFit UC-07,Streetwear,Everyday,M,purple,4409.37,3289.390020
2,3,SaigonFlex UC-59,Streetwear,Everyday,L,white,42.908735,24.500483
3,4,HanoiStreet UM-36,Streetwear,Balanced,S,orange,9109.17,8653.711500
4,5,SaigonFlex UC-97,Streetwear,Everyday,S,orange,4844.7,2618.075880


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    999 non-null    object 
 1   product_name  1000 non-null   object 
 2   category      1000 non-null   object 
 3   segment       1000 non-null   object 
 4   size          1000 non-null   object 
 5   color         1000 non-null   object 
 6   price         1000 non-null   object 
 7   cogs          1000 non-null   float64
dtypes: float64(1), object(7)
memory usage: 62.6+ KB


In [10]:
# Chuyển product_id sang Int64 (xử lý NaN)
epd["product_id"] = pd.to_numeric(epd["product_id"], errors="coerce").astype("Int64")
# Kiểm tra các giá trị product_id bị NaN
display(epd[epd["product_id"].isna()])

,product_id,product_name,category,segment,size,color,price,cogs
31,<NA>,DragonWear UE-02,Streetwear,Performance,L,white,8386.030435,7570.069673
287,<NA>,SaigonFlex UM-46,Streetwear,Balanced,M,black,24.208376,14.981117


In [11]:
# Ta sẽ kiêm tra trong project đã có sản phẩm này chưa nếu chưa ta sẽ drop 2 dòng này đi
missing = epd[epd["product_id"].isna()].reset_index()
cols = [
    "product_name", "category", "segment",
    "size", "color", "price", "cogs"
]
reference = products[cols].copy()
for col in ["price", "cogs"]:
    missing[col] = pd.to_numeric(missing[col], errors="coerce").round(6)
    reference[col] = pd.to_numeric(reference[col], errors="coerce").round(6)

matched = missing.merge(reference.drop_duplicates(), on=cols, how="inner")
epd = epd.drop(index=matched["index"])

In [12]:
# Kiểm tra các dòng trùng ID
display(epd[epd["product_id"].duplicated(keep=False)])

,product_id,product_name,category,segment,size,color,price,cogs
403,405,BambooCraft UC-25,Streetwear,Everyday,L,green,8808.87,7972.027350
404,405,VietMode MA-20,Casual,All-weather,S,white,60.901002,38.018708


In [13]:
# Có 2 dòng bị trùng product_id nhưng khác giá trị khác nên ta giữ và xem cũng đa có trong prodcuts chưa
# nếu có thì drop.
duplicate = epd[epd["product_id"].duplicated(keep=False)].reset_index()
cols = [
    "product_name", "category", "segment",
    "size", "color", "price", "cogs"
]
reference = products[cols].copy()
for col in ["price", "cogs"]:
    duplicate[col] = pd.to_numeric(duplicate[col], errors="coerce").round(6)
    reference[col] = pd.to_numeric(reference[col], errors="coerce").round(6)

matched = duplicate.merge(reference.drop_duplicates(), on=cols, how="inner")
epd = epd.drop(index=matched["index"])
display(epd[epd["product_id"].duplicated(keep=False)])

,product_id,product_name,category,segment,size,color,price,cogs


In [14]:
# Xem các giá trị danh mục
print(epd["category"].value_counts(dropna=False))

category
Streetwear     535
Outdoor        304
Casual          90
GenZ            65
N/A              1
Unknown_999      1
Name: count, dtype: int64


In [15]:
# Xem thử 2 giá trị N/A và Unknown_999
display(epd[epd["category"].isnull() | epd["category"].isin(["N/A", "Unknown_999"])])

,product_id,product_name,category,segment,size,color,price,cogs
529,530,MekongFit UE-20,N/A,Performance,S,white,7559.37,5095.771317
812,813,SaigonFlex UM-96,Unknown_999,Balanced,XL,black,11060.080545,10507.076517


In [16]:
# Lấy các dòng category bị lỗi
missing = epd[
    epd["category"].isin(["Unknown_999", "N/A"])
].reset_index()

# Các trường dùng để đối chiếu
cols = ["product_name", "segment", "size", "color", "price", "cogs"]

reference = products[cols].copy()

for col in ["price", "cogs"]:
    missing[col] = pd.to_numeric(missing[col], errors="coerce").round(6)
    reference[col] = pd.to_numeric(reference[col], errors="coerce").round(6)

# Tìm các dòng đã có trong products
matched = missing.merge(
    reference.drop_duplicates(),
    on=cols,
    how="inner"
)

display(matched)

# Chỉ drop các dòng đã khớp
epd = epd.drop(index=matched["index"])

# Kiểm tra còn category lỗi không
display(epd[epd["category"].isin(["Unknown_999", "N/A"])])

,index,product_id,product_name,category,segment,size,color,price,cogs
0,529,530,MekongFit UE-20,N/A,Performance,S,white,7559.370000,5095.771317
1,812,813,SaigonFlex UM-96,Unknown_999,Balanced,XL,black,11060.080545,10507.076517


,product_id,product_name,category,segment,size,color,price,cogs


In [17]:
# Xem các giá trị segment
print(epd["segment"].value_counts(dropna=False))

segment
Activewear     244
Everyday       164
Performance    140
Balanced       128
Standard       103
All-weather     79
Premium         71
Trendy          65
Name: count, dtype: int64


In [18]:
# Xem các giá trị size
print(epd["size"].value_counts(dropna=False))

size
M     266
S     264
XL    236
L     228
Name: count, dtype: int64


In [19]:
# Xem các giá trị color
print(epd["color"].value_counts(dropna=False))

color
silver    114
red       108
pink      103
yellow    101
orange     99
white      98
purple     97
black      95
blue       95
green      84
Name: count, dtype: int64


In [20]:
# Chuyển price sang numeric, kiểm tra NaN
epd["price"] = pd.to_numeric(epd["price"], errors="coerce")
display(epd[epd["price"].isna()])

,product_id,product_name,category,segment,size,color,price,cogs
193,194,SaigonFlex UC-84,Streetwear,Everyday,XL,yellow,NaN,6509.61927


In [21]:
# Lấy các dòng thiếu price
missing = epd[epd["price"].isna()].reset_index()

cols = ["product_name", "category", "segment", "size", "color", "cogs"]
reference = products[cols].copy()

# Làm tròn giá vốn để so sánh
missing["cogs"] = pd.to_numeric(
    missing["cogs"], errors="coerce"
).round(6)

reference["cogs"] = pd.to_numeric(
    reference["cogs"], errors="coerce"
).round(6)

# Tìm các dòng đã có trong products
matched = missing.merge(
    reference.drop_duplicates(),
    on=cols,
    how="inner"
)

display(matched)

# Bỏ các dòng đã khớp
epd = epd.drop(index=matched["index"])

# Kiểm tra còn dòng thiếu giá không
display(epd[epd["price"].isna()])

,index,product_id,product_name,category,segment,size,color,price,cogs
0,193,194,SaigonFlex UC-84,Streetwear,Everyday,XL,yellow,NaN,6509.61927


,product_id,product_name,category,segment,size,color,price,cogs


In [22]:
# Check giá vốn và giá bán bị âm
display(epd[epd["price"] < 0])
display(epd[epd["cogs"] < 0])

,product_id,product_name,category,segment,size,color,price,cogs
72,73,HanoiStreet RP-60,Outdoor,Activewear,S,orange,-150.0,1835.397459


,product_id,product_name,category,segment,size,color,price,cogs


In [23]:
# Lấy các dòng có giá bán âm
invalid = epd[epd["price"] < 0].reset_index()

cols = ["product_name", "category", "segment", "size", "color", "cogs"]
reference = products[cols].copy()

# Chuẩn hóa giá vốn để so sánh
invalid["cogs"] = pd.to_numeric(
    invalid["cogs"], errors="coerce"
).round(6)

reference["cogs"] = pd.to_numeric(
    reference["cogs"], errors="coerce"
).round(6)

# Tìm các dòng đã có trong products
matched = invalid.merge(
    reference.drop_duplicates(),
    on=cols,
    how="inner"
)

display(matched)

# Chỉ drop các dòng đã khớp
epd = epd.drop(index=matched["index"])

# Kiểm tra còn dòng giá âm không
display(epd[epd["price"] < 0])

,index,product_id,product_name,category,segment,size,color,price,cogs
0,72,73,HanoiStreet RP-60,Outdoor,Activewear,S,orange,-150.0,1835.397459


,product_id,product_name,category,segment,size,color,price,cogs


In [24]:
# Reset index
epd = epd.reset_index(drop=True)
print(f'✅ EPD sạch: {len(epd)} dòng')
epd.info()

✅ EPD sạch: 992 dòng
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 992 entries, 0 to 991
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    992 non-null    Int64  
 1   product_name  992 non-null    object 
 2   category      992 non-null    object 
 3   segment       992 non-null    object 
 4   size          992 non-null    object 
 5   color         992 non-null    object 
 6   price         992 non-null    float64
 7   cogs          992 non-null    float64
dtypes: Int64(1), float64(2), object(5)
memory usage: 63.1+ KB


## PHẦN 3: Chuẩn hoá kiểu dữ liệu trước khi merge
Chuẩn hoá kiểu dữ liệu 2 bên để tránh bị lệch khi so sánh:
- `product_id`: Int64
- `product_name, category, segment, size, color`: str (strip + lowercase)
- `price, cogs`: float64

In [25]:
# Chuẩn hóa Products
products["product_id"] = products["product_id"].astype("Int64")

for col in ["product_name", "category", "segment", "size", "color"]:
    products[col] = products[col].astype("string").str.strip().str.lower()

products["price"] = pd.to_numeric(products["price"], errors="coerce")
products["cogs"] = pd.to_numeric(products["cogs"], errors="coerce")


# Chuẩn hóa EPD
epd["product_id"] = pd.to_numeric(
    epd["product_id"], errors="coerce"
).astype("Int64")

for col in ["product_name", "category", "segment", "size", "color"]:
    epd[col] = epd[col].astype("string").str.strip().str.lower()

epd["price"] = pd.to_numeric(epd["price"], errors="coerce")
epd["cogs"] = pd.to_numeric(epd["cogs"], errors="coerce")

In [26]:
# Kiểm tra dtype đã khớp nhau chưa
print('Kiểm tra dtype khớp:')
for col in products.columns:
    p_type = str(products[col].dtype)
    e_type = str(epd[col].dtype)
    match = '✅' if p_type == e_type else '❌'
    print(f'  {col:15s} | Products: {p_type:10s} | EPD: {e_type:10s} | {match}')

Kiểm tra dtype khớp:
  product_id      | Products: Int64      | EPD: Int64      | ✅
  product_name    | Products: string     | EPD: string     | ✅
  category        | Products: string     | EPD: string     | ✅
  segment         | Products: string     | EPD: string     | ✅
  size            | Products: string     | EPD: string     | ✅
  color           | Products: string     | EPD: string     | ✅
  price           | Products: float64    | EPD: float64    | ✅
  cogs            | Products: float64    | EPD: float64    | ✅


In [27]:
# Chuẩn hoá lại cột price và cogs
products["cogs_check"] = products["cogs"].round(6)
epd["cogs_check"] = epd["cogs"].round(6)


## PHẦN 4: Merge & So sánh tất cả cột (trừ product_id)
- So sánh EPD vs Products trên **tất cả cột trừ `product_id`**
- Nếu sản phẩm EPD đã có trong Products → **bỏ qua**
- Chỉ thêm sản phẩm **thật sự mới** từ EPD

In [28]:
# Các cột dùng để so sánh: TẤT CẢ trừ product_id
compare_cols = matching_cols = [
    "product_name",
    "segment",
    "size",
    "color",
    "cogs_check"
]
print(f'Cột so sánh: {compare_cols}')

# Merge trên tất cả cột trừ product_id
merged = pd.merge(
    epd, products,
    on=compare_cols,
    how='left',
    indicator=True,
    suffixes=('_epd', '_products')
)

# _merge == 'both'      → đã tồn tại trong products → BỎ QUA
# _merge == 'left_only'  → chưa có → THÊM VÀO
already_exists = merged[merged['_merge'] == 'both']
new_from_epd = merged[merged['_merge'] == 'left_only']

print(f'\n📊 Kết quả so sánh EPD vs Products:')
print(f'   EPD tổng cộng            : {len(epd)} dòng')
print(f'   Đã có trong Products (bỏ): {len(already_exists)} dòng')
print(f'   Thật sự mới (thêm vào)   : {len(new_from_epd)} dòng')

Cột so sánh: ['product_name', 'segment', 'size', 'color', 'cogs_check']

📊 Kết quả so sánh EPD vs Products:
   EPD tổng cộng            : 992 dòng
   Đã có trong Products (bỏ): 992 dòng
   Thật sự mới (thêm vào)   : 0 dòng


In [29]:
# Kết luận là không cần gộp bỏ luôn file epd.json
# Drop cột cogs_check
products = products.drop(columns=["cogs_check"])

In [30]:
# Xem thông tin cuối cùng
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   Int64  
 1   product_name  2412 non-null   string 
 2   category      2412 non-null   string 
 3   segment       2412 non-null   string 
 4   size          2412 non-null   string 
 5   color         2412 non-null   string 
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: Int64(1), float64(2), string(5)
memory usage: 153.2 KB


In [31]:
products.isna().sum()

product_id      0
product_name    0
category        0
segment         0
size            0
color           0
price           0
cogs            0
dtype: int64

In [32]:
# Xuất dữ liệu ra file CSV duy nhất
products.to_csv('../SilverData/products_silver.csv', index=False)
